# 从CLIP到BLIP-2————Q-Former作为模态桥梁

CLIP将图片和文本对齐，但是不能够做标题生产，回答问题。BLIP-2的解决方式是一个小的可训练的桥梁————32个可学习的查询向量，通过交叉注意力观察被冻结的ViT特征，然后将其作为冻结LLM层的输入。

## 问题描述

假设ViT中一个Patch嵌入到了1408维度，大语言模型的输入维度是4096，显然易见的桥梁是一个将1408映射到4096的线性层，但是把所有的Patch都放到LLM的上下问中就是Patch数量个额外Token，还会随批次大小增长。

BLIP-2的做法是将Patch数量个Token压缩成更少的Token，同时保留足够的信息让LLM完成下游任务。产物就是Q-Former，32个可学习的“query（Q）”向量，通过交叉注意力总结所有的视觉特征，然后送入LLM。

## 基本概念

### 可学习的查询向量

Q-Former的核心把戏是：不让LLM关注所有的Patch Token，而是引入一个中间层，32个可学习的查询向量，让他们来关注Patch Token。

经过交叉注意力之后，每个查询向量都包含图像的压缩摘要。

### 架构

Q-Former是一个小的Transformer（12层，大概100M参数）：
- 查询路径。  32个查询向量各自之间做自注意力，然后与冻结的ViT Patch Token做交叉注意力。
- 文本路径。  一个像BERT的文本编码器，共享查询路径的的自注意力和前馈权重（Query向量会被拼在Text前面，所以Text注意Query使用的注意力权重与Query自注意Query的权重一致）。文本的交叉注意力被禁用（不注意ViT Patches）。

训练的时候两条路径都在跑。query和text通过自注意力交互。

### 两阶段训练

BLIP-2的训练分为两个阶段：

第一阶段，表征学习，没有LLM参与。损失有三项：
- ITC （Image-Text Contrastive）：CLIP风格的对比损失，在池化的Query Token 和文本 CLS Token上计算。
- ITM （Image-Text matching）： 二分类器，文本图像是否成对。做难负例挖掘。
- ITG （Image-grounded text generation）：文本上的因果LM头，以query为条件，逼着query编码文本生产相关的内容。
这个阶段只有Q-Former在训练，ViT冻结，没有LLM参与。

第二阶段：生成式学习。接上一个冻结的LLM。将32个查询输出通过一个小的线性层接入到LLM的嵌入维度，假装它们是文本的嵌入维度。根据提示词+图像+标题 序列只训练线性投影层以及Q-Former。

训练完后，Q-Former和投影层就是全部的视觉适配器。在推理阶段，图像过ViT，经Q-Former和线性投影假装成文本输入，最后LLM据此生成输出。

### InstructBLIP 和 指令驱动的Q-Former

为Q-Former拓展了一个额外输入————指令文本。在交叉注意力的时候，查询向量现在能够同时注意到图像Patch和这条指令。

### MiniGPT-4 只有 project层

MiniGPT-4 保留了Q-Former，但是只训练输出投影层，其他的全部冻结。牺牲质量换预算，查询向量的结构不是你自己的。

### LLaVA 更简单

直接将Q-Former替换成了一个2层感知机，把每个ViT Patch Token投影到LLM空间，压缩率很低，不过现在模型能够看到原生的Patch了。它在后续被证明可以被训练成保留足够的信号，因而是有效的。取舍是：LLaVA的上下文窗口消耗得更快，但是可以自然拓展到多图像和视频。

所以在预算紧张的地方Q-Former占主导。在以单Token质量为先的地方MLP投影器占主导。

### 门控交叉注意力：Flamingo

Flamingo早于BLIP-2，在每个冻结的LLM层使用交叉注意力，而不是作为单座桥梁。BLIP-2表明你可以只压缩到输入层，也能工作。两者结合，上下文中会有少样本例子（in-context few-shot）。

# 开始编码

教学积木：小尺寸 Q-Former 关键（不复刻 12 层真模型）。


## 1. 可学习 Query + 注意力积木

Query 是参数；ViT patch 当交叉注意力的 K/V；文本不做对 patch 的交叉注意。


In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyConfig:
    """故意用小尺寸，方便打印 shape、看懂数据流。"""
    num_queries: int = 8          # 真 BLIP-2 常用 32
    dim: int = 64                 # Q-Former 隐层
    vit_dim: int = 48             # 冻结 ViT 的 patch 维
    llm_dim: int = 96             # 下游 LLM 嵌入维
    n_heads: int = 4
    num_patches: int = 16         # 假 ViT：比如 4x4 patches
    vocab_size: int = 50
    max_text_len: int = 12
    n_layers: int = 2             # 真模型约 12


class LearnableQueries(nn.Module):
    """(1, Nq, D) 可学习 query；forward 时按 batch expand。"""

    def __init__(self, num_queries: int, dim: int):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, num_queries, dim) * 0.02)

    def forward(self, batch_size: int) -> torch.Tensor:
        return self.query.expand(batch_size, -1, -1)


class SelfAttention(nn.Module):
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor | None = None) -> torch.Tensor:
        # attn_mask: (L, L) True/float 屏蔽位；这里用 float mask（0 可见，-inf 屏蔽）
        h, _ = self.attn(x, x, x, attn_mask=attn_mask, need_weights=False)
        return self.norm(x + h)


class CrossAttention(nn.Module):
    """Q 来自 query，K/V 来自冻结 ViT patch。文本路径禁用这个模块。"""

    def __init__(self, dim: int, vit_dim: int, n_heads: int):
        super().__init__()
        self.kv_proj = nn.Linear(vit_dim, dim)  # 把 ViT 维接到 Q-Former 维
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, query: torch.Tensor, vit_feats: torch.Tensor) -> torch.Tensor:
        kv = self.kv_proj(vit_feats)
        h, _ = self.attn(query, kv, kv, need_weights=False)
        return self.norm(query + h)


class FFN(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x + self.net(x))


print("积木定义完成:", LearnableQueries, SelfAttention, CrossAttention)


## 2. Q-Former Block + 三种注意力 Mask

ITC：query/text 互不看；ITM：双向互看；ITG：text 因果，且可看全部 query。


In [ ]:
class QFormerBlock(nn.Module):
    """
    单层示意：
    - query: SelfAttn -> CrossAttn(ViT) -> FFN
    - text : SelfAttn(可与 query 拼接后做) -> FFN  （无 CrossAttn）
    共享一套 self-attn / FFN 权重，贴近笔记里「共享自注意力和前馈」。
    """

    def __init__(self, cfg: TinyConfig):
        super().__init__()
        self.self_attn = SelfAttention(cfg.dim, cfg.n_heads)
        self.cross_attn = CrossAttention(cfg.dim, cfg.vit_dim, cfg.n_heads)
        self.ffn = FFN(cfg.dim)

    def forward_queries(self, q: torch.Tensor, vit_feats: torch.Tensor) -> torch.Tensor:
        q = self.self_attn(q)
        q = self.cross_attn(q, vit_feats)
        return self.ffn(q)

    def forward_joint(
        self,
        q: torch.Tensor,
        t: torch.Tensor,
        vit_feats: torch.Tensor,
        joint_mask: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """把 [query | text] 拼起来做共享自注意力，再拆开；仅 query 做交叉注意。"""
        nq = q.size(1)
        joint = torch.cat([q, t], dim=1)
        joint = self.self_attn(joint, attn_mask=joint_mask)
        q_out, t_out = joint[:, :nq], joint[:, nq:]
        q_out = self.cross_attn(q_out, vit_feats)
        q_out = self.ffn(q_out)
        t_out = self.ffn(t_out)
        return q_out, t_out


def make_itc_mask(nq: int, nt: int, device) -> torch.Tensor:
    """ITC：两块单模态。mask[i,j]=0 可见，-inf 屏蔽。"""
    L = nq + nt
    mask = torch.zeros(L, L, device=device)
    # query 不能看 text
    mask[:nq, nq:] = float("-inf")
    # text 不能看 query
    mask[nq:, :nq] = float("-inf")
    return mask


def make_itm_mask(nq: int, nt: int, device) -> torch.Tensor:
    """ITM：全可见（双向）。"""
    return torch.zeros(nq + nt, nq + nt, device=device)


def make_itg_mask(nq: int, nt: int, device) -> torch.Tensor:
    """
    ITG：
    - query 互看，不看 text（生成时视觉摘要先独立）
    - text 可看全部 query + 因果看上文 text
    """
    L = nq + nt
    mask = torch.zeros(L, L, device=device)
    # query 不看 text
    mask[:nq, nq:] = float("-inf")
    # text 因果：位置 j 不能看 j 之后的 text
    for i in range(nt):
        # row = nq+i，不能看 text 里 > i 的位置
        if i + 1 < nt:
            mask[nq + i, nq + i + 1 :] = float("-inf")
    return mask


def demo_masks(nq: int = 3, nt: int = 4):
    print("ITC mask (0=可见, -inf=屏蔽):")
    print(make_itc_mask(nq, nt, "cpu"))
    print("\nITM mask:")
    print(make_itm_mask(nq, nt, "cpu"))
    print("\nITG mask:")
    print(make_itg_mask(nq, nt, "cpu"))


demo_masks()


## 3. 拼装 TinyQFormer + 三损失玩具 + LLM 投影

第一阶段：ITC / ITM / ITG 逼 query；第二阶段示意：query → Linear → 假装成 LLM token。


In [ ]:
class TinyQFormer(nn.Module):
    def __init__(self, cfg: TinyConfig):
        super().__init__()
        self.cfg = cfg
        self.queries = LearnableQueries(cfg.num_queries, cfg.dim)
        self.text_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers = nn.ModuleList([QFormerBlock(cfg) for _ in range(cfg.n_layers)])
        # ITC: 对比头
        self.vision_proj = nn.Linear(cfg.dim, cfg.dim)
        self.text_proj = nn.Linear(cfg.dim, cfg.dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
        # ITM: 匹配头
        self.itm_head = nn.Linear(cfg.dim, 2)
        # ITG: 因果 LM 头
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size)

    def embed_text(self, text_ids: torch.Tensor) -> torch.Tensor:
        return self.text_emb(text_ids)

    def encode(
        self,
        vit_feats: torch.Tensor,
        text_ids: torch.Tensor,
        mode: str,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """mode in {itc, itm, itg}，决定 joint mask。"""
        B = vit_feats.size(0)
        q = self.queries(B)
        t = self.embed_text(text_ids)
        nq, nt = q.size(1), t.size(1)
        if mode == "itc":
            mask = make_itc_mask(nq, nt, vit_feats.device)
        elif mode == "itm":
            mask = make_itm_mask(nq, nt, vit_feats.device)
        elif mode == "itg":
            mask = make_itg_mask(nq, nt, vit_feats.device)
        else:
            raise ValueError(mode)

        for layer in self.layers:
            q, t = layer.forward_joint(q, t, vit_feats, mask)
        return q, t

    def loss_itc(self, vit_feats: torch.Tensor, text_ids: torch.Tensor) -> torch.Tensor:
        q, t = self.encode(vit_feats, text_ids, "itc")
        # 池化 query；文本用第 0 个位置当假 CLS
        v = F.normalize(self.vision_proj(q.mean(dim=1)), dim=-1)
        u = F.normalize(self.text_proj(t[:, 0]), dim=-1)
        logits = v @ u.T * self.logit_scale.exp().clamp(max=100)
        labels = torch.arange(v.size(0), device=v.device)
        return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

    def loss_itm(self, vit_feats: torch.Tensor, text_ids: torch.Tensor) -> torch.Tensor:
        # 玩具难负样本：batch 内滚一下文本
        B = vit_feats.size(0)
        q_pos, t_pos = self.encode(vit_feats, text_ids, "itm")
        neg_ids = torch.roll(text_ids, shifts=1, dims=0)
        q_neg, t_neg = self.encode(vit_feats, neg_ids, "itm")

        # 用 [CLS 文本位] 与 query 均值拼一下再分类（简化）
        def score(q, t):
            h = q.mean(dim=1) + t[:, 0]
            return self.itm_head(h)

        logits = torch.cat([score(q_pos, t_pos), score(q_neg, t_neg)], dim=0)
        labels = torch.cat(
            [
                torch.ones(B, dtype=torch.long, device=vit_feats.device),
                torch.zeros(B, dtype=torch.long, device=vit_feats.device),
            ]
        )
        return F.cross_entropy(logits, labels)

    def loss_itg(self, vit_feats: torch.Tensor, text_ids: torch.Tensor) -> torch.Tensor:
        # 输入 text[:, :-1]，预测 text[:, 1:]；query 经 mask 条件化文本
        inp = text_ids[:, :-1]
        tgt = text_ids[:, 1:]
        q, t = self.encode(vit_feats, inp, "itg")
        logits = self.lm_head(t)  # (B, Nt-1, V)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))


class QueryToLLMProj(nn.Module):
    """第二阶段桥梁：32(这里8) 个 query -> LLM 嵌入维，假装成文本 token。"""

    def __init__(self, dim: int, llm_dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, llm_dim)

    def forward(self, query_out: torch.Tensor) -> torch.Tensor:
        return self.proj(query_out)


class FrozenToyLLM(nn.Module):
    """冻结 LLM 示意：只证明 soft visual prompt 能拼进序列。"""

    def __init__(self, vocab_size: int, llm_dim: int):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, llm_dim)
        self.block = nn.TransformerEncoderLayer(
            d_model=llm_dim, nhead=4, dim_feedforward=llm_dim * 2, batch_first=True
        )
        self.head = nn.Linear(llm_dim, vocab_size)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, visual_soft_tokens: torch.Tensor, prompt_ids: torch.Tensor) -> torch.Tensor:
        text = self.tok(prompt_ids)
        x = torch.cat([visual_soft_tokens, text], dim=1)
        x = self.block(x)
        return self.head(x)


print("TinyQFormer / QueryToLLMProj / FrozenToyLLM ready")


## 4. 冒烟测试：shape + 一步三损失 + 投影进 LLM


In [ ]:
def smoke_test():
    torch.manual_seed(0)
    cfg = TinyConfig()
    device = "cpu"
    model = TinyQFormer(cfg).to(device)
    proj = QueryToLLMProj(cfg.dim, cfg.llm_dim).to(device)
    llm = FrozenToyLLM(cfg.vocab_size, cfg.llm_dim).to(device)

    B = 4
    # 假冻结 ViT 输出
    vit_feats = torch.randn(B, cfg.num_patches, cfg.vit_dim)
    text_ids = torch.randint(1, cfg.vocab_size, (B, cfg.max_text_len))
    prompt_ids = torch.randint(1, cfg.vocab_size, (B, 5))

    print("=== shapes ===")
    q, t = model.encode(vit_feats, text_ids, "itm")
    print(f"vit_feats: {tuple(vit_feats.shape)}")
    print(f"queries out: {tuple(q.shape)}  # (B, Nq, D)")
    print(f"text out   : {tuple(t.shape)}  # (B, Nt, D)")

    soft = proj(q)
    print(f"LLM soft tokens: {tuple(soft.shape)}  # (B, Nq, D_llm)")
    llm_logits = llm(soft, prompt_ids)
    print(f"LLM logits: {tuple(llm_logits.shape)}  # (B, Nq+Nprompt, V)")

    print("\n=== stage-1 toy losses (one step) ===")
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3
    )
    opt.zero_grad()
    litc = model.loss_itc(vit_feats, text_ids)
    litm = model.loss_itm(vit_feats, text_ids)
    litg = model.loss_itg(vit_feats, text_ids)
    loss = litc + litm + litg
    loss.backward()
    opt.step()
    print(f"ITC={litc.item():.4f}  ITM={litm.item():.4f}  ITG={litg.item():.4f}  sum={loss.item():.4f}")

    # 确认 ViT 输入未要求梯度（我们传的是叶子张量且未设 requires_grad）
    print(f"vit_feats.requires_grad={vit_feats.requires_grad}  (模拟冻结 ViT 特征)")
    print(f"trainable Q-Former params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
    print(f"trainable proj params   : {sum(p.numel() for p in proj.parameters() if p.requires_grad)}")
    print(f"LLM params frozen       : {sum(p.numel() for p in llm.parameters() if not p.requires_grad)}")
    print("SMOKE TEST OK")


smoke_test()
